In [13]:
import wandb
import pandas as pd
import os
from tqdm.notebook import tqdm

# from table_plotter import print_result_table

In [14]:
api = wandb.Api(timeout=600)


In [15]:
# Specify cache directory
cache_dir = "./wandb_cache"
os.makedirs(cache_dir, exist_ok=True)

In [16]:

skipped_runs = []  # List to store IDs of skipped runs

evaluation_keys = ['Evaluation/acc_imp_perc', 'Evaluation/exist_imp_perc', 'Evaluation/reach_imp_perc', 'Evaluation/path_length',
                   'Evaluation/fn_imp_perc', 'Evaluation/fp_imp_perc', 'Evaluation/tn_imp_perc', 'Evaluation/tp_imp_perc', 
                   'Evaluation/solvability', 'Evaluation/playability']
evaluation2_keys = ['Evaluation/playability', 'Evaluation/naive_playability', 'Evaluation/solvability', 'Evaluation/acc_imp_perc']


In [17]:
def get_dataframe_from_run(run):
    dfs = list()
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 'n_aux_worst', 'n_aux_best',
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [18]:
def get_dataframe_from_run2(run):
    dfs = []
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation2_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation2_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 'n_aux_worst', 'n_aux_best',
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation2_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [19]:
runs = api.runs("inchangbaek4907/scenario-aux")
scenario_df = get_dataframe_from_run(runs)
scenario_df = scenario_df[scenario_df['Evaluation/llm_iteration'] <= 6]
# set score column with acc_imp_perc
scenario_df['score'] = scenario_df['Evaluation/acc_imp_perc']
scenario_df

  0%|          | 0/70 [00:00<?, ?it/s]

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/exist_imp_perc,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score
0,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.633333,0.166667,0.00000,3.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
1,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.666667,0.016667,0.00000,3.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
2,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.716667,0.250000,26.00000,2.9,0.0,0.033333,0.066667,0.033333,0.033333,0.033333
3,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.400000,0.250000,26.90000,2.8,0.0,0.133333,0.066667,0.133333,0.666667,0.066667
4,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.633333,0.150000,30.00000,3.0,0.0,0.000000,0.000000,0.000000,0.066667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b2w2_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.950000,0.533333,26.50000,3.0,0.0,0.000000,0.000000,0.000000,0.133333,0.000000
416,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b2w2_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.983333,0.383333,26.00000,3.0,0.0,0.000000,0.000000,0.000000,0.033333,0.000000
417,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b2w2_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.983333,0.383333,26.00000,3.0,0.0,0.000000,0.000000,0.000000,0.033333,0.000000
418,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b2w2_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.983333,0.383333,26.00000,3.0,0.0,0.000000,0.000000,0.000000,0.033333,0.000000


In [20]:
runs = api.runs("inchangbaek4907/scenario2-aux")
scenario2_df = get_dataframe_from_run2(runs)
scenario2_df = scenario2_df[scenario2_df['Evaluation/llm_iteration'] <= 6]
scenario2_df['score'] = scenario2_df['Evaluation/playability']
scenario2_df

  0%|          | 0/70 [00:00<?, ?it/s]

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,reward_feature,fewshot,problem,seed,Evaluation/llm_iteration,Evaluation/playability,Evaluation/naive_playability,Evaluation/solvability,Evaluation/acc_imp_perc,score
0,p38hcx46,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
1,p38hcx46,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,3,2,0.000000,0.000000,0.000000,0.000000,0.000000
2,p38hcx46,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,3,3,0.033333,0.033333,0.200000,0.400000,0.033333
3,p38hcx46,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,3,4,0.000000,0.600000,0.600000,0.900000,0.000000
4,p38hcx46,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,3,5,0.000000,0.600000,0.600000,0.900000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,2,2,0.533333,0.533333,0.600000,0.633333,0.533333
409,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,2,3,0.066667,0.166667,0.233333,0.366667,0.066667
410,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,2,4,0.066667,0.166667,0.233333,0.366667,0.066667
411,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,array,False,dungeon4,2,5,0.066667,0.166667,0.233333,0.366667,0.066667


In [21]:
scenario_df = pd.concat([scenario_df, scenario2_df], ignore_index=True)
scenario_df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.166667,0.0,3.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
1,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.016667,0.0,3.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.250000,26.0,2.9,0.0,0.033333,0.066667,0.033333,0.033333,0.033333,NaN
3,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.250000,26.9,2.8,0.0,0.133333,0.066667,0.133333,0.666667,0.066667,NaN
4,pe-got_it-6_fit-hr_exp-aux_t-sce_aux-b1w0_chr-...,finished,2,got,gpt-4o,2,aux,hr,6,0,...,0.150000,30.0,3.0,0.0,0.000000,0.000000,0.000000,0.066667,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
828,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.600000,0.533333,0.533333,0.533333
829,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.233333,0.066667,0.066667,0.166667
830,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.233333,0.066667,0.066667,0.166667
831,32lxrdij,finished,5,got,gpt-4o,2,aux,hr,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.233333,0.066667,0.066667,0.166667


In [22]:
# Print summary of skipped runs
print("\nSummary of Skipped Runs:")
print(f"Total skipped runs: {len(skipped_runs)}")
print("Skipped run IDs:", skipped_runs)


Summary of Skipped Runs:
Total skipped runs: 0
Skipped run IDs: []


In [23]:
df = pd.concat([scenario_df], ignore_index=True)

In [24]:
df.to_csv(f"aux_result.csv", index=False)